# Waveform susceptibility in anisotropic Womersley flow

This Google Colab notebook mounts Google Drive, creates a unique run directory, installs the repository software, executes the complete six-artery analysis, and writes reproducible tables, arrays, checksums, and publication figures. The figure suite uses a common two-column width, sans-serif typography, lower-case panel labels, unified colour scales, SI units, vector output, and 600 dpi raster output.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from uuid import uuid4

REPOSITORY_URL = 'https://github.com/khalid-saqr/picoNewton.git'
REPOSITORY_REF = os.environ.get('PICONEWTON_REF', 'main')
SESSION_ID = (
    datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    + '_'
    + uuid4().hex[:8]
)
LOCAL_ROOT = Path('/content') / f'picoNewton_{SESSION_ID}'
RUN_ROOT = (
    Path('/content/drive/MyDrive/picoNewton_waveform_susceptibility/runs')
    / SESSION_ID
)
ANALYSIS_ROOT = RUN_ROOT / 'analysis'
RUN_ROOT.mkdir(parents=True, exist_ok=False)

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
subprocess.run(
    ['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(LOCAL_ROOT)],
    check=True,
)
subprocess.run(['git', 'checkout', REPOSITORY_REF], cwd=LOCAL_ROOT, check=True)
COMMIT_SHA = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=LOCAL_ROOT, text=True
).strip()
print('Run directory:', RUN_ROOT)
print('Repository commit:', COMMIT_SHA)


In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(str(item) for item in command))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-e',
    str(LOCAL_ROOT / 'picoNewton_v3'),
])
run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-e',
    str(LOCAL_ROOT / 'waveform_susceptibility'),
])


In [ ]:
metadata = {
    'session_id': SESSION_ID,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'repository_url': REPOSITORY_URL,
    'repository_ref': REPOSITORY_REF,
    'commit_sha': COMMIT_SHA,
    'python': sys.version,
    'platform': platform.platform(),
    'analysis_root': str(ANALYSIS_ROOT),
}
(RUN_ROOT / 'runtime_metadata.json').write_text(
    json.dumps(metadata, indent=2, sort_keys=True),
    encoding='utf-8',
)
metadata


In [ ]:
run([
    'piconewton-waveform-susceptibility',
    '--output',
    str(ANALYSIS_ROOT),
    '--radial-order',
    '150',
    '--time-points',
    '2048',
    '--quadrature-nodes',
    '256',
    '--validation-epsilon',
    '0.08',
    '--figure-dpi',
    '600',
])


In [ ]:
import hashlib
import zipfile

required = {
    'analysis_summary.json',
    'artery_atlas.csv',
    'crossed_susceptibility.csv',
    'waveform_controls.csv',
    'harmonic_pair_attribution.csv',
    'reduced_law_validation.csv',
    'constitutive_robustness.csv',
    'operator_archive.npz',
    'figures/figure_manifest.json',
    'figures/figure_manifest.csv',
}
missing = sorted(name for name in required if not (ANALYSIS_ROOT / name).is_file())
assert not missing, f'Missing outputs: {missing}'

figure_manifest = json.loads(
    (ANALYSIS_ROOT / 'figures/figure_manifest.json').read_text(encoding='utf-8')
)
assert figure_manifest['figure_count'] == 6
assert figure_manifest['common_width_mm'] == 180.0
assert figure_manifest['maximum_height_mm'] <= 170.0
assert figure_manifest['font_size_pt'] == 7.0
assert figure_manifest['minimum_line_width_pt'] >= 1.0
for extension in ('pdf', 'svg', 'png'):
    assert len(list((ANALYSIS_ROOT / 'figures').glob(f'figure_[1-6]_*.{extension}'))) == 6

checksum_lines = []
for path in sorted(item for item in RUN_ROOT.rglob('*') if item.is_file()):
    if path.name in {'checksums.sha256', 'waveform_susceptibility_results.zip'}:
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksum_lines.append(f'{digest}  {path.relative_to(RUN_ROOT).as_posix()}')
(RUN_ROOT / 'checksums.sha256').write_text(
    '\n'.join(checksum_lines) + '\n', encoding='utf-8'
)

archive_path = RUN_ROOT / 'waveform_susceptibility_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(item for item in RUN_ROOT.rglob('*') if item.is_file()):
        if path == archive_path:
            continue
        archive.write(path, path.relative_to(RUN_ROOT))
print('Verified output directory:', RUN_ROOT)
print('Archive:', archive_path)


In [ ]:
from IPython.display import Image, display

summary = json.loads(
    (ANALYSIS_ROOT / 'analysis_summary.json').read_text(encoding='utf-8')
)
display(summary)
for figure in sorted((ANALYSIS_ROOT / 'figures').glob('figure_[1-6]_*.png')):
    display(Image(filename=str(figure), width=1100))
